# 05 — ML Baseline (Global): Data Load & Alignment

**Goal:** Load the curated LINCS L1000 Vitamin D subset, align expression (`X`) with signature metadata (`meta`), and run sanity checks before modeling.


### Imports

In [ ]:
import os
import json
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, ElasticNet
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, r2_score, mean_absolute_error


Paths

In [ ]:
# === Input paths (edit to match your repo) ===
EXP_PATH = "../data/exports/expression_matrix_clean.parquet"   # genes x signatures (columns = sig_id)
META_PATH = "../data/exports/signature_metadata_clean.csv"    # rows = signatures, includes 'sig_id'

assert os.path.exists(EXP_PATH), f"Missing file: {EXP_PATH}"
assert os.path.exists(META_PATH), f"Missing file: {META_PATH}"

print("EXP_PATH:", EXP_PATH)
print("META_PATH:", META_PATH)


Load

In [ ]:
exp = pd.read_parquet(EXP_PATH)   # expected: rows=genes, cols=sig_id
meta = pd.read_csv(META_PATH)     # expected: one row per sig_id

print("Expression matrix shape (genes x signatures):", exp.shape)
print("Metadata shape (rows x cols):", meta.shape)

display(meta.head(3))


Required columns + alignment

In [ ]:
# --- Required columns ---
required_cols = ["sig_id", "cell_id", "pert_dose"]
missing = [c for c in required_cols if c not in meta.columns]
assert not missing, f"Metadata is missing required columns: {missing}"

# --- Keep only signatures present in the expression matrix ---
meta = meta[meta["sig_id"].isin(exp.columns)].copy()

# --- Align expression columns to metadata order ---
exp = exp.loc[:, meta["sig_id"].tolist()]

# --- Sanity checks ---
assert exp.shape[1] == meta.shape[0], "Mismatch: exp columns != meta rows after alignment"
assert meta["sig_id"].is_unique, "Metadata contains duplicated sig_id values"
assert exp.columns.is_unique, "Expression matrix contains duplicated signature columns"

print("Aligned expression shape (genes x signatures):", exp.shape)
print("Aligned metadata shape:", meta.shape)
print("Unique cell lines:", meta["cell_id"].nunique())


Missingness / basic QC

In [ ]:
# Check missingness in expression
sig_all_nan = exp.isna().all(axis=0).sum()
gene_all_nan = exp.isna().all(axis=1).sum()
total_nan = int(exp.isna().sum().sum())

print(f"Signatures with all-NaN expression: {sig_all_nan}")
print(f"Genes with all-NaN expression: {gene_all_nan}")
print(f"Total NaNs in expression matrix: {total_nan}")

# Quick metadata QC
print("Dose summary:")
display(meta["pert_dose"].describe())

print("Dose (min, max):", float(meta["pert_dose"].min()), float(meta["pert_dose"].max()))


## Step 2 — Build ML-ready inputs (X, y, groups)

We construct:
- `X`: signature-level feature matrix (samples × genes)
- `y`: binary dose label (`dose_bin`: low vs high), defined **within each cell line**
- `groups`: cell line grouping variable for leakage-safe cross-validation


## Create dose_bin within each cell line
This avoids “cheating” due to global dose scale differences between cell lines.

In [ ]:
# Create a within-cell-line binary dose label using the median dose per cell line
# Rule: dose >= median(cell_id) -> "high", else "low"

meta = meta.copy()

if "dose_bin" not in meta.columns:
    meta["dose_bin"] = (
        meta.groupby("cell_id")["pert_dose"]
        .transform(lambda s: np.where(s >= s.median(), "high", "low"))
    )

# Encode y as 0/1 (high=1)
y = (meta["dose_bin"].astype(str).str.lower() == "high").astype(int).to_numpy()

# Quick check: class balance overall and by cell line
overall_rate = float(y.mean())
print(f"Overall positive rate (high dose): {overall_rate:.3f}")

by_cell = (
    meta.assign(y=y)
    .groupby("cell_id")["y"]
    .agg(n="count", pos="sum", pos_rate="mean")
    .sort_values("n", ascending=False)
)
display(by_cell)


## Build `X` (samples × genes) and groups

In [ ]:
# Build feature matrix X: samples are signatures, features are genes
# exp: genes x signatures -> X: signatures x genes
X = exp.T.to_numpy()

# Group labels for leakage-safe CV (leave-one-cell-line-out)
groups = meta["cell_id"].astype(str).to_numpy()

print("X shape (n_samples x n_genes):", X.shape)
print("y shape:", y.shape)
print("groups shape:", groups.shape)
print("Unique groups:", np.unique(groups))

Sanity checks (alignment + missingness)

In [ ]:
# Alignment checks
assert X.shape[0] == meta.shape[0] == y.shape[0] == groups.shape[0], "Row mismatch in X/meta/y/groups"

# Missingness check (genes are columns in X)
nan_per_sample = np.isnan(X).mean(axis=1)
nan_per_gene = np.isnan(X).mean(axis=0)

print("NaN rate per sample (min/median/max):",
      float(np.min(nan_per_sample)),
      float(np.median(nan_per_sample)),
      float(np.max(nan_per_sample)))

print("NaN rate per gene (min/median/max):",
      float(np.min(nan_per_gene)),
      float(np.median(nan_per_gene)),
      float(np.max(nan_per_gene)))

# If there are NaNs, we will handle them explicitly before modeling (next step)
n_nan_total = int(np.isnan(X).sum())
print("Total NaNs in X:", n_nan_total)


### Notes

- We defined `dose_bin` within each `cell_id` to avoid confounding by between-cell-line dose distributions.
- Next, we will define a simple, leakage-safe baseline model using:
  - `StandardScaler`
  - `PCA`
  - `LogisticRegression`
  evaluated with **leave-one-cell-line-out** cross-validation (`GroupKFold`).


## Step 3 — Global baseline model (PCA + Logistic Regression)

We evaluate whether gene expression profiles contain signal to discriminate
low vs high dose conditions across cell lines.

Validation strategy:
- Leave-one-cell-line-out cross-validation (GroupKFold)
- Metrics: ROC AUC, Average Precision, Accuracy


Define pipeline

In [ ]:
pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=50, random_state=42)),
        ("clf", LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )),
    ]
)


Cross-validation

In [ ]:
cv = GroupKFold(n_splits=len(np.unique(groups)))

scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "accuracy": "accuracy",
}

cv_results = cross_validate(
    pipe,
    X,
    y,
    cv=cv.split(X, y, groups=groups),
    scoring=scoring,
    return_train_score=False,
)

# Summarize results
summary = (
    pd.DataFrame(cv_results)
    .filter(like="test_")
    .agg(["mean", "std"])
    .T
)

summary


### Interpretation

This baseline evaluates global dose-related signal while enforcing
generalization across cell lines.

Performance above random expectation (ROC AUC > 0.5) indicates
that gene expression profiles encode dose-dependent information
shared across cellular contexts.

A global linear baseline achieves ROC AUC ≈ 0.71 under leave-one-cell-line-out CV, indicating shared dose-dependent transcriptional signal across cellular contexts.

## Step 4 — Minimal baseline: core_score only

We compare the full-transcriptome model against a minimal,
biologically informed baseline using only the Vitamin D core score.

This evaluates whether genome-wide expression adds predictive value
beyond the predefined core signature.


## Step 3.5 — Recompute core_score

The core score is recomputed locally to keep this notebook self-contained.
Definition:

core_score = mean(z(core_UP genes)) − mean(z(core_DN genes))


Paste core gene lists

In [ ]:
# Core gene lists (defined in directed analysis)
CORE_UP_GENES = [54541, 26227, 4864, 25987, 3280, 976, 
                  6616, 4792, 16, 25, 80212, 1978, 5106, 
                  6696, 3486, 9181, 7296, 1277, 8821, 10628, 
                  6509, 12, 55107, 1831, 2274, 3638, 10276, 
                  2770, 10551, 6919, 25966, 11197, 58472, 
                  1050, 1111, 178, 1870, 23097, 10227, 644,
                  4067, 23635
                  ]

CORE_DN_GENES = [11098, 5111, 79080, 10112, 10362, 6192, 
                  51116, 2280, 11137, 5909, 998, 6277, 
                  6347, 9128, 9060, 840, 6697, 2745, 10049, 
                  4502, 3157, 54205, 2697, 9270, 23029, 27242, 
                  7846, 51635, 91949, 8870, 1465, 5993, 3206, 
                  6659, 9518
                  ]

print("N core UP genes:", len(CORE_UP_GENES))
print("N core DN genes:", len(CORE_DN_GENES))


Compute core_score

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Ensure genes are present
core_up = [g for g in CORE_UP_GENES if g in exp.index]
core_dn = [g for g in CORE_DN_GENES if g in exp.index]

print(f"Core UP genes found: {len(core_up)} / {len(CORE_UP_GENES)}")
print(f"Core DN genes found: {len(core_dn)} / {len(CORE_DN_GENES)}")

assert len(core_up) > 0 and len(core_dn) > 0, "Core gene lists do not match expression index"

# Z-score per gene across signatures (row-wise standardization)
# StandardScaler standardizes features column-wise, so we transpose twice.
scaler = StandardScaler(with_mean=True, with_std=True)

exp_z = pd.DataFrame(
    scaler.fit_transform(exp.T).T,
    index=exp.index,
    columns=exp.columns
)

# Compute core score per signature
core_score = exp_z.loc[core_up].mean(axis=0) - exp_z.loc[core_dn].mean(axis=0)

# Attach to metadata (aligned by sig_id)
meta["core_score"] = meta["sig_id"].map(core_score)

# Final sanity checks
assert meta["core_score"].notna().all(), "Some sig_id values were not matched in core_score"

display(meta["core_score"].describe())


## Step 4 — Minimal baseline: core_score only

We evaluate whether the biologically defined Vitamin D core score
alone can discriminate low vs high dose conditions, under the same
cross-cell-line validation strategy.


Build X_core

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_validate

# Feature matrix: single biologically informed feature
X_core = meta["core_score"].to_numpy().reshape(-1, 1)

print("X_core shape:", X_core.shape)


CV evaluation

In [ ]:
pipe_core = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )),
    ]
)

cv = GroupKFold(n_splits=len(np.unique(groups)))

scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "accuracy": "accuracy",
}

cv_core = cross_validate(
    pipe_core,
    X_core,
    y,
    cv=cv.split(X_core, y, groups=groups),
    scoring=scoring,
    return_train_score=False,
)

summary_core = (
    pd.DataFrame(cv_core)
    .filter(like="test_")
    .agg(["mean", "std"])
    .T
)

summary_core


## Step 6 — ElasticNet regression to predict core_score

We model the Vitamin D core score as a continuous outcome using
ElasticNet regression on genome-wide expression profiles.

This approach enables feature selection while accounting for
correlated gene expression patterns.


Define target and CV groups
### Modeling choices

- Target: `core_score` (continuous)
- Features: gene expression (full transcriptome)
- Regularization: ElasticNet (L1 + L2)
- Validation: leave-one-cell-line-out cross-validation


In [ ]:
# Regression target
y_reg = meta["core_score"].to_numpy()

# Group labels (cell lines) for leakage-safe CV
groups_reg = meta["cell_id"].astype(str).to_numpy()

print("Target shape:", y_reg.shape)
print("Unique groups:", np.unique(groups_reg))


Base pipeline (no tuning yet)

In [ ]:
enet_pipe = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("enet", ElasticNet(
            alpha=1.0,
            l1_ratio=0.5,
            max_iter=10000,
            random_state=42
        )),
    ]
)


### Notes

- Scaling is mandatory before ElasticNet.
- Hyperparameters (`alpha`, `l1_ratio`) will be tuned via nested CV.
- Model interpretability will focus on coefficient stability,
  not single-fit coefficients.


## Step 7 — Nested cross-validation for ElasticNet regression

We use nested cross-validation to obtain an unbiased estimate of
predictive performance when tuning ElasticNet hyperparameters.

- Outer loop: leave-one-cell-line-out (GroupKFold)
- Inner loop: GridSearchCV on training folds
- Metrics: R² and Mean Absolute Error (MAE)


In [ ]:
# Hyperparameter grid for ElasticNet
param_grid = {
    "enet__alpha": np.logspace(-3, 1, 9),   # regularization strength
    "enet__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9],
}

param_grid


In [ ]:
inner_cv = 5

grid = GridSearchCV(
    estimator=enet_pipe,
    param_grid=param_grid,
    scoring="r2",
    cv=inner_cv,
    n_jobs=-1,
)


In [ ]:
outer_cv = GroupKFold(n_splits=len(np.unique(groups_reg)))

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
}

cv_nested = cross_validate(
    grid,
    X,
    y_reg,
    cv=outer_cv.split(X, y_reg, groups=groups_reg),
    scoring=scoring,
    return_estimator=True,
    n_jobs=-1,
)

# Summarize performance
nested_summary = (
    pd.DataFrame(cv_nested)
    .filter(like="test_")
    .assign(test_mae=lambda df: -df["test_mae"])
    .agg(["mean", "std"])
    .T
)

nested_summary


### Interpretation

- R² quantifies the proportion of variance in `core_score`
  explained by transcriptomic features.
- MAE provides an absolute error scale, easier to interpret
  in the units of `core_score`.
- Stability across outer folds indicates robustness across
  cellular contexts.


## Step 8 — Coefficient extraction and stability analysis

We extract ElasticNet coefficients from each outer cross-validation fold
to assess feature selection stability and directionality.


In [ ]:
# Extract coefficients from each outer fold
coefs = []

for i, est in enumerate(cv_nested["estimator"]):
    best_enet = est.best_estimator_.named_steps["enet"]
    coef = best_enet.coef_
    
    coefs.append(
        pd.Series(coef, index=exp.index, name=f"fold_{i}")
    )

coef_df = pd.concat(coefs, axis=1)

print("Coefficient matrix shape:", coef_df.shape)
coef_df.head()


Each column corresponds to ElasticNet coefficients fitted on one
outer training fold. Rows correspond to genes.
